In [1]:
import json


with open('templates/template_sequences.json', 'r') as f:
    data = json.load(f)

print(len(data.keys()))

20448


In [5]:
seq2pdb = {}
for key, value in data.items():
    for chain, seq in value.items():
        if seq not in seq2pdb:
            seq2pdb[seq] = {}
        seq2pdb[seq][key[:4]] = chain

print(len(seq2pdb))

for seq, chain2pdb in seq2pdb.items():
    if len(chain2pdb) > 1:
        print(seq, len(chain2pdb), chain2pdb)



36628
MASPLDQAIGLLIGIFHKYSGKEGDKHTLSKKELKELIQKELTIGSKLQDAEIVKLMDDLDRNKDQEVNFQEYITFLGALAMIYNEALKG 3 {'1a03': 'B', '1cnp': 'B', '2jtt': 'B'}
IVEGQDAEVGLSPWQVMLFRKSPQELLCGASLISDRWVLTAAHCLLYPPWDKNFTVDDLLVRIGKHSRTRYERKVEKISMLDKIYIHPRYNWKENLDRDIALLKLKRPIELSDYIHPVCLPDKQTAAKLLHAGFKGRVTGWGNRRETWTTSVAEVQPSVLQVVNLPLVERPVCKASTRIRITDNMFCAGYKPGEGKRGDACEGDSGGPFVMKSPYNNRWYQMGIVSWGEGCDRDGKYGFYTHVFRLKKWIQKVIDRLGS 6 {'1a0h': 'E', '1avg': 'H', '1tbq': 'K', '1toc': 'H', '1ucy': 'N', '1vit': 'H'}
ADKELKFLVVDDFSTMRRIVRNLLKELGFNNVEEAEDGVDALNKLQAGGYGFVISDWNMPNMDGLELLKTIRADGAMSALPVLMVTAEAKKENIIAAAQAGASGYVVKPFTAATLEEKLNKIFEKLGM 5 {'1a0o': 'G', '1bdj': 'A', '1eay': 'B', '1f4v': 'B', '1kmi': 'Y'}
RDFNNLTKGLCTINSWHIYGKDNAVRIGEDSDVLVTREPYVSCDPDECRFYALSQGTTIRGKHSNGTIHDRSQYRALISWPLSSPPTVYNSRVECIGWSSTSCHDGKTRMSICISGPNNNASAVIWYNRRPVTEINTWARNILRTQESECVCHNGVCPVVFTDGSATGPAETRIYYFKEGKILKWEPLAGTAKHIEECSCYGERAEITCTCRDNWQGSNRPVIRIDPVAMTHTSQYICSPVLTDNPRPNDPTVGKCNDPYPGNNNNGVKGFSYLDGVNTWLGRTISIASRSGYEMLKVPNALTDDKSKPTQGQTIVLNTDWSG

In [8]:
import os

from Bio.PDB import PDBParser, NeighborSearch, PPBuilder

# List of pdb:chain mappings
pdb_chains = {'1a03': 'B', '1cnp': 'B', '2jtt': 'B'}

contact_residues = {}

pdb_folder = 'templates/pdbs'

# Helper: Only consider atoms in residue ('CA' for alpha carbon; you can later sample all heavy atoms if needed)
def get_contact_residues(structure, chain_id, distance_cutoff=5.0):
    # Get specified chain
    model = list(structure.get_models())[0]
    chain = model[chain_id]
    # All residues from the chain
    residues = list(chain.get_residues())
    # All atoms for NeighborSearch
    atoms = [atom for atom in structure.get_atoms()]
    ns = NeighborSearch(atoms)
    contacts = set()
    # For each residue in chain, check if any atom is within cutoff of an atom in a different chain
    for residue in residues:
        # Just in case skip HETATM, water, etc.
        if not hasattr(residue, 'id') or residue.id[0] != ' ':
            continue
        for atom in residue.get_atoms():
            near_atoms = ns.search(atom.coord, distance_cutoff, level='A')
            for near in near_atoms:
                parent_residue = near.get_parent()
                parent_chain = parent_residue.get_parent()
                # Only count contacts from other chains
                if parent_chain.id != chain_id:
                    # Save residue number (seq id) in contact
                    contacts.add((residue.get_id()[1], residue.get_resname()))
                    break  # No need to check further atoms for this residue
    return contacts

for pdb_id, chain_id in pdb_chains.items():
    pdb_path = os.path.join(pdb_folder, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    contacts = get_contact_residues(structure, chain_id)
    contact_residues[pdb_id] = contacts

# Print contact residues
for pdb_id, contacts in contact_residues.items():
    print(f"PDB {pdb_id}, contact residues in chain {pdb_chains[pdb_id]}:")
    print(contacts)
    # for resnum, resname in sorted(contacts):
    #     print(f"  - Residue {resname} {resnum}")
    # print()

PDB 1a03, contact residues in chain B:
{(45, 'GLY'), (75, 'THR'), (44, 'ILE'), (78, 'GLY'), (14, 'GLY'), (13, 'ILE'), (27, 'HIS'), (3, 'SER'), (15, 'ILE'), (42, 'LEU'), (39, 'GLN'), (85, 'ASN'), (86, 'GLU'), (4, 'PRO'), (11, 'LEU'), (88, 'LEU'), (1, 'MET'), (8, 'ALA'), (48, 'LEU'), (77, 'LEU'), (83, 'ILE'), (28, 'THR'), (19, 'TYR'), (74, 'ILE'), (89, 'LYS'), (20, 'SER'), (37, 'LEU'), (69, 'ASN'), (43, 'THR'), (87, 'ALA'), (70, 'PHE'), (25, 'ASP'), (24, 'GLY'), (90, 'GLY'), (51, 'ALA'), (76, 'PHE'), (82, 'MET'), (18, 'LYS'), (40, 'LYS'), (41, 'GLU'), (46, 'SER'), (52, 'GLU'), (56, 'LEU'), (12, 'LEU'), (55, 'LYS'), (80, 'LEU'), (5, 'LEU'), (60, 'LEU'), (49, 'GLN'), (17, 'HIS'), (71, 'GLN'), (38, 'ILE'), (7, 'GLN'), (23, 'GLU'), (10, 'GLY'), (73, 'TYR'), (84, 'TYR'), (6, 'ASP'), (36, 'GLU'), (53, 'ILE'), (29, 'LEU'), (26, 'LYS'), (2, 'ALA'), (16, 'PHE'), (79, 'ALA'), (21, 'GLY'), (9, 'ILE'), (81, 'ALA')}
PDB 1cnp, contact residues in chain B:
{(45, 'GLY'), (75, 'THR'), (44, 'ILE'), (78, '